## 1.0 Libraries and directories

In [6]:
import os

os.chdir('/Users/jmaze/Documents/projects/green-by-another-name/')

In [7]:
import ee 
import geemap
import geopandas as gpd
import datetime as dt

import pprint as pp

ykf = gpd.read_file('./data/roi_shapes/YKF_roi_shape.shp')

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

roi_ee = ee.Geometry(ykf.iloc[0].geometry.__geo_interface__)

# Original dates (save these)
date = '2020-05-29'
date_plus1d = '2020-05-30'


Enter verification code:  4/1ASVgi3KgYOJJLDDyAeNB4dY6Qrc8edhzxFOtDgEtEaCVRqKKRFdJx42uvjI



Successfully saved authorization token.


## 2.0 Collection of Sentinel-2 Images on target date

In [8]:
s2_spec_reflec = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
    .filterBounds(roi_ee)
    .filterDate(date, date_plus1d)
)

s2_spec_reflec = s2_spec_reflec.select(['B2', 'B3', 'B4', 'B8', 'B8A', 'SCL'])
s2_spec_reflec = s2_spec_reflec.mosaic()
s2_spec_reflec = s2_spec_reflec.clip(roi_ee)


## 3.0 Mask the poor quality Sentinel-2 pixels

In [30]:
s2_cl_prob = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
    .filterBounds(roi_ee)
    .filterDate(date, date_plus1d)
)
s2_cl_prob = s2_cl_prob.mosaic()
s2_cl_prob = s2_cl_prob.clip(roi_ee)

s2_cl_mask = s2_cl_prob.select('probability').gt(20).rename('cl_binary')
s2_dark_mask = s2_spec_reflec.select('SCL').eq(2)
s2_cloud_shaddow_mask = s2_spec_reflec.select('SCL').eq(3)
s2_cirrus_mask = s2_spec_reflec.select('SCL').eq(10)
#s2_opaque_clouds_mask = s2_spec_reflec.select('QA60').bitwiseAnd(1 << 10).eq(0)

s2_full_mask = s2_cl_mask.Or(s2_cloud_shaddow_mask).Or(s2_cirrus_mask)

## 4.0 Polygon for Sentinel-2 tile's extent

In [4]:

s2_data_mask = s2_spec_reflec.mask().reduce(ee.Reducer.anyNonZero())
s2_boundary = s2_data_mask.reduceToVectors(
    geometry=roi_ee,
    geometryType='polygon',
    scale=10,
    maxPixels=1e13
)

s2_boundary = ee.Feature(s2_boundary.toList(s2_boundary.size()).get(0))


## 5.0 Collection of Landsat Images on target date

In [5]:
def optical_rescale(img):
    "Only converts optical bands, thermal bands not included"
    img = img.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5']
    )
    img = img.multiply(0.0000275).add(-0.2)

    return img


ls_spec_reflec = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                  .filterBounds(roi_ee)
                  .filterDate(date, date_plus1d)
)

# pp.pp(ls_spec_reflec.getInfo())

ls_qa = ls_spec_reflec.select('QA_PIXEL').mosaic()
ls_qa = ls_qa.clip(roi_ee)

ls_spec_reflec = ls_spec_reflec.select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5'])
ls_spec_reflec = ls_spec_reflec.map(optical_rescale)

ls_total_imgs = ls_spec_reflec.size().getInfo()
ls_spec_reflec = ls_spec_reflec.mosaic()
ls_spec_reflec = ls_spec_reflec.clip(roi_ee)


## 6.0 Mask the poor quality Landsat pixels

In [28]:

# Dilated clouds over land and water
ls_dilated_cloud_ovr_land = ls_qa.eq(21826)
ls_dilated_cloud_ovr_wtr = ls_qa.eq(21890)

ls_full_mask = ls_dilated_cloud_ovr_land.Or(ls_dilated_cloud_ovr_wtr)

# Mid confidence clouds
ls_mid_conf_cloud = ls_qa.eq(22080)
ls_mid_conf_cloud_ovr_wtr = ls_qa.eq(22144)
ls_mid_conf_cloud_wshaddow = ls_qa.eq(24088)
ls_mid_conf_cloud_wshaddow_ovr_wtr = ls_qa.eq(24216)

ls_full_mask = (
    ls_full_mask.Or(ls_mid_conf_cloud)
    .Or(ls_mid_conf_cloud_ovr_wtr)
    .Or(ls_mid_conf_cloud_wshaddow)
    .Or(ls_mid_conf_cloud_wshaddow_ovr_wtr)
)

# High confidence clouds
ls_high_conf_cloud = ls_qa.eq(22280)
ls_high_conf_cloud_wshaddow = ls_qa.eq(24344)

ls_full_mask = (
    ls_full_mask.Or(ls_high_conf_cloud)
    .Or(ls_high_conf_cloud_wshaddow)
)

# Shaddows
ls_high_conf_shaddow = ls_qa.eq(23888)
ls_wtr_w_cloud_shaddow = ls_qa.eq(23952)

ls_full_mask = (
    ls_full_mask.Or(ls_high_conf_shaddow)
    .Or(ls_wtr_w_cloud_shaddow)
)

# Cirrus
ls_high_conf_cirrus = ls_qa.eq(54596)
ls_cirrus_mid_cloud = ls_qa.eq(54852)
ls_cirrus_high_cloud = ls_qa.eq(55052)

ls_full_mask = (
    ls_full_mask.Or(ls_high_conf_cirrus)
    .Or(ls_cirrus_mid_cloud)
    .Or(ls_cirrus_high_cloud)
)

# Snow/ice
ls_snow_ice = ls_qa.eq(30048)

ls_full_mask = ls_full_mask.Or(ls_snow_ice)

## 7.0 Polygon for Landsat tile's extent

In [13]:
# Get the bounds of the Landsat image

ls_data_mask = ls_spec_reflec.mask().reduce(ee.Reducer.anyNonZero())
ls_boundary = ls_data_mask.reduceToVectors(
    geometry=roi_ee,
    geometryType='polygon',
    scale=10,
    maxPixels=1e13
)

ls_boundary = ee.Feature(ls_boundary.toList(ls_boundary.size()).get(2))

## 8.0 Visualize the results

In [14]:
s2_true_col_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.7
}

s2_nir_r_g = {
    'bands': ['B8', 'B4', 'B3'],
    'min': 0, 
    'max': 3000, 
    
}

ls_true_col_params = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0,
    'max': 0.3,
    'gamma': 1.7
}

s2_cloud_prob_params = {
    'bands': ['probability'],
    'min': 0,
    'max': 100
}

binary_mask_red_params = {
    'min':0,
    'max':1,
    'palette': ['grey', 'red']
}

binary_mask_white_params = {
    'min':0,
    'max':1, 
    'palette': ['grey', 'white']
}
binary_mask_orange_params = {
    'min':0,
    'max':1, 
    'palette': ['grey', 'orange']
}

binary_mask_lblue_params = {
    'min':0,
    'max':1, 
    'palette': ['grey', '#87CEFA']
}


In [34]:
Map = geemap.Map()

Map.addLayer(s2_spec_reflec, s2_true_col_params, f'S2 True Color')
#Map.addLayer(s2_spec_reflec, s2_nir_r_g, 'S2 False Color')
#Map.addLayer(ls_spec_reflec, ls_true_col_params, f'LS8 True Color')

#Map.addLayer(s2_cl_prob, s2_cloud_prob_params, 'S2 Cloud Probability')
# Map.addLayer(s2_cl_mask, binary_mask_white_params, 'S2 Cloud Mask')
# Map.addLayer(s2_cloud_shaddow_mask, binary_mask_red_params, 'S2 Shaddow Mask')
#Map.addLayer(s2_dark_mask, binary_mask_orange_params, 'S2 Dark Mask')
# Map.addLayer(s2_cirrus_mask, binary_mask_lblue_params, 'S2 Cirrus Mask')
Map.addLayer(s2_full_mask, binary_mask_red_params, 'S2 Full Mask')


#Map.addLayer(ls_full_mask, binary_mask_red_params, 'LS Full QA Mask')
#Map.addLayer(ls_dilated_cloud_ovr_land, binary_mask_red_params, 'LS Full QA Mask')
#Map.addLayer(ls_mid_conf_cloud, binary_mask_orange_params, 'Mid Conf')
#Map.addLayer(ls_high_conf_cloud, binary_mask_lblue_params, 'High Conf')


#Map.addLayer(s2_boundary, {'color': 'blue'}, 'S2 Boundary')
#Map.addLayer(ls_boundary, {'color': 'green'}, 'LS Boundary')
#Map.addLayer(roi_ee, {'color': 'red'}, 'ROI Outline')
Map.centerObject(roi_ee, zoom=10)
Map


Map(center=[66.51859072213907, -146.00018537875346], controls=(WidgetControl(options=['position', 'transparent…

## 9.0 Export data masks and images

In [10]:
s2_img_out = s2_spec_reflec.select(['B2', 'B3', 'B4', 'B8']).clip(ls_boundary) 
# Clip the S2 image becuase ls is larger
ls_img_out = ls_spec_reflec.select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5'])

In [ ]:
s2_full_mask = s2_full_mask.clip(ls_boundary)

In [ ]:
# ls_export = ee.batch.Export.image.toDrive(
#     image=ls_img_out,
#     description='ls-img-test',
#     fileNamePrefix='ls-img-test',
#     folder='ls_s2_coincidents', 
#     scale=30,
#     region=roi_ee,
#     crs='EPSG:4326',
#     fileFormat='GeoTIFF',
#     maxPixels=1e13
# )

# # Start the export task
# ls_export.start()

In [31]:
# ls_mask_export = ee.batch.Export.image.toDrive(
#     image=ls_full_mask,
#     description='ls-mask-test-v2',
#     fileNamePrefix='ls-masks-test-v2',
#     folder='ls_s2_coincidents_masks', 
#     scale=30,
#     region=roi_ee,
#     crs='EPSG:4326',
#     fileFormat='GeoTIFF',
#     maxPixels=1e13
# )

# # Start the export task
# ls_mask_export.start()

In [10]:
# s2_export = ee.batch.Export.image.toDrive(
#     image=s2_img_out,
#     description='s2-img-res10m',
#     fileNamePrefix='s2-img-res10m',
#     folder='ls_s2_coincidents', 
#     scale=10,
#     region=roi_ee,
#     crs='EPSG:4326',
#     fileFormat='GeoTIFF',
#     maxPixels=1e13
# )

# # Start the export task
# s2_export.start()

In [12]:
est_roi_utm = gpd.read_file('./data/YKflats_roi_shape.shp').estimate_utm_crs()

print(est_roi_utm)

EPSG:32606


In [14]:
target_proj = 'EPSG:32606'
target_scale = 30

s2_img_reproj = s2_img_out.reproject(
    crs=target_proj,
    scale=target_scale
)

s2_img_reproj_reduced = s2_img_reproj.reduceResolution(
    reducer=ee.Reducer.mean()
).reproject(
    crs=s2_img_out.projection(),
    scale=30
)
    

In [20]:
s2_img_coarse = s2_img_reproj.resample('bilinear').reproject(
    crs=s2_img_out.projection(),
    scale=30
)

In [22]:
# s2_resampled_export = ee.batch.Export.image.toDrive(
#     image=s2_img_coarse,
#     description='s2-img-reproj-resampled',
#     fileNamePrefix='s2-img-reproj-resampled',
#     folder='ls_s2_coincidents', 
#     scale=30,
#     region=roi_ee,
#     crs='EPSG:4326',
#     fileFormat='GeoTIFF',
#     maxPixels=1e13
# )

# # Start the export task
# s2_resampled_export.start()

In [ ]:
s2_mask_export = ee.batch.Export.image.toDrive(
    image=s2_full_mask,
    description='s2-mask-test-v2',
    fileNamePrefix='s2-mask-test-v2',
    folder='ls_s2_coincidents_masks', 
    scale=10,
    region=roi_ee,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e13
)

# Start the export task
s2_mask_export.start()

## Export the image boundaries

In [12]:
s2_boundary_fc = ee.FeatureCollection(s2_boundary)
ls_boundary_fc = ee.FeatureCollection(ls_boundary)

In [14]:
# export_polygon = ee.batch.Export.table.toDrive(
#     collection=ls_boundary_fc,
#     description='ls-boundary',
#     fileFormat='SHP',    
#     fileNamePrefix='ls-boundary',  # Prefix for the exported file
# )

# # Start the export task
# export_polygon.start()

## 